In [1]:
# Enables IPython autoreload (two magic commands, the second takes a numeric argument)
%load_ext autoreload
%autoreload 2

import logging
import os
import time

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s  - %(name)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.6f}".format)


logger.info("Notebook initialized")

2026-04-30 14:04:06,561  - __main__ - INFO - Notebook initialized


---

## Final pipeline using `src/data.py`

The above cells walked through the exploration step-by-step. The same pipeline is now packaged into two functions in `src/data.py`. The cells below verify that the imported functions produce the same output as the inline exploration.

In [2]:
from src.data import compute_returns, download_prices

prices = download_prices()
returns = compute_returns(prices)

print("Daily prices shape:", prices.shape)
print("Monthly returns shape:", returns.shape)
print("\nFirst 3 returns:")
print(returns.head(3))
print("\nLast 3 returns:")
print(returns.tail(3))

2026-04-30 14:04:06,603  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-04-30 14:04:06,681  - src.data - INFO - Computed monthly returns: (120, 8)


Daily prices shape: (2538, 8)
Monthly returns shape: (120, 8)

First 3 returns:
                 SPY      GOVT     EEMV       CME       BR      CBOE  \
date                                                                   
2015-01-31 -0.029629  0.029423 0.011831 -0.037789 0.039194  0.016556   
2015-02-28  0.056205 -0.017439 0.025305  0.124619 0.109189 -0.065708   
2015-03-31 -0.015745  0.006021 0.004426 -0.007544 0.038856 -0.043728   

                 ICE       ACN  
date                            
2015-01-31 -0.061836 -0.059120  
2015-02-28  0.144024  0.071403  
2015-03-31 -0.006068  0.040653  

Last 3 returns:
                 SPY      GOVT      EEMV      CME        BR      CBOE  \
date                                                                    
2024-10-31 -0.008924 -0.024286 -0.037161 0.021346 -0.019393  0.042466   
2024-11-30  0.059634  0.008489 -0.008945 0.056088  0.119321  0.013626   
2024-12-31 -0.024100  0.006945 -0.007585 0.004852 -0.038463 -0.094742   

           

In [3]:
from src.data import compute_returns, download_prices
from src.stats import arithmetic_mean, geometric_mean

prices = download_prices()
returns = compute_returns(prices)

print("Arithmetic mean (monthly):")
print(arithmetic_mean(returns))
print("\nGeometric mean (monthly):")
print(geometric_mean(returns))

2026-04-30 14:04:06,703  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-04-30 14:04:06,714  - src.data - INFO - Computed monthly returns: (120, 8)


Arithmetic mean (monthly):
SPY    0.011216
GOVT   0.000917
EEMV   0.002965
CME    0.012852
BR     0.016831
CBOE   0.012521
ICE    0.013060
ACN    0.015002
dtype: float64

Geometric mean (monthly):
SPY    0.010243
GOVT   0.000815
EEMV   0.002326
CME    0.011446
BR     0.014829
CBOE   0.010574
ICE    0.011327
ACN    0.012883
dtype: float64


In [4]:
from src.data import compute_returns, download_prices
from src.stats import arithmetic_mean, covariance_matrix, geometric_mean, volatility

prices = download_prices()
returns = compute_returns(prices)

print("Arithmetic mean (monthly):")
print(arithmetic_mean(returns))
print("\nVolatility (monthly):")
print(volatility(returns))
print("\nCovariance matrix (monthly):")
print(covariance_matrix(returns))
print("\nDiagonal of covariance vs volatility squared:")
print((volatility(returns) ** 2).round(8))
print(np.diag(covariance_matrix(returns)).round(8))

2026-04-30 14:04:06,745  - src.data - INFO - Loading prices from cache: /Users/tangentjet/Projects/mean-variance-portfolio/data/raw/prices_daily.parquet
2026-04-30 14:04:06,750  - src.data - INFO - Computed monthly returns: (120, 8)


Arithmetic mean (monthly):
SPY    0.011216
GOVT   0.000917
EEMV   0.002965
CME    0.012852
BR     0.016831
CBOE   0.012521
ICE    0.013060
ACN    0.015002
dtype: float64

Volatility (monthly):
SPY    0.044245
GOVT   0.014359
EEMV   0.035842
CME    0.053643
BR     0.064040
CBOE   0.062143
ICE    0.059594
ACN    0.065292
dtype: float64

Covariance matrix (monthly):
          SPY      GOVT     EEMV       CME       BR     CBOE      ICE      ACN
SPY  0.001958  0.000083 0.001100  0.000858 0.001865 0.000937 0.001755 0.002302
GOVT 0.000083  0.000206 0.000106 -0.000023 0.000229 0.000036 0.000168 0.000183
EEMV 0.001100  0.000106 0.001285  0.000315 0.000910 0.000341 0.000783 0.001126
CME  0.000858 -0.000023 0.000315  0.002878 0.001094 0.001688 0.001821 0.001110
BR   0.001865  0.000229 0.000910  0.001094 0.004101 0.001098 0.001957 0.002774
CBOE 0.000937  0.000036 0.000341  0.001688 0.001098 0.003862 0.001688 0.001430
ICE  0.001755  0.000168 0.000783  0.001821 0.001957 0.001688 0.003551 0.002368
AC